In [1]:
# Project Title

#**Logistic Regression Analysis of High Billing Amount**

## Research Question

#For this assignment, I examined whether **patient Age** was associated with the odds of having a **high billing amount**, after adjusting for **Gender** and **Length of Stay**.

#The binary response variable was created as:

#- **0 = Billing Amount less than $25,000**
#- **1 = Billing Amount $25,000 or more**

#Age was the primary explanatory variable. Gender and Length of Stay were included as potential confounding variables.

## Python Program

#```python
import pandas as pd
import numpy as np
import statsmodels.api as sm
import glob
import os

# ============================================================
# FIND AND LOAD DATA
# ============================================================

csv_files = glob.glob("**/*.csv", recursive=True)

data_file = None

for file in csv_files:
    if os.path.basename(file).lower() == "healthcare_dataset.csv":
        data_file = file
        break

if data_file is None:
    for file in csv_files:
        if "healthcare" in os.path.basename(file).lower():
            data_file = file
            break

if data_file is None:
    raise FileNotFoundError(
        "healthcare_dataset.csv was not found. "
        "Please upload the dataset first."
    )

data = pd.read_csv(data_file, low_memory=False)

print("=" * 75)
print("LOGISTIC REGRESSION: HIGH BILLING AMOUNT")
print("=" * 75)

# ============================================================
# DATA MANAGEMENT
# ============================================================

# Age
data["Age_Clean"] = pd.to_numeric(
    data["Age"],
    errors="coerce"
)

data.loc[
    (data["Age_Clean"] < 0) |
    (data["Age_Clean"] > 120),
    "Age_Clean"
] = np.nan

# Billing Amount
data["Billing_Clean"] = pd.to_numeric(
    data["Billing Amount"],
    errors="coerce"
)

data.loc[
    data["Billing_Clean"] <= 0,
    "Billing_Clean"
] = np.nan

# Create binary response variable
data["High_Billing"] = np.nan

valid_billing = data["Billing_Clean"].notna()

data.loc[
    valid_billing,
    "High_Billing"
] = (
    data.loc[
        valid_billing,
        "Billing_Clean"
    ] >= 25000
).astype(int)

# Length of Stay
data["Admission_Date"] = pd.to_datetime(
    data["Date of Admission"],
    errors="coerce"
)

data["Discharge_Date"] = pd.to_datetime(
    data["Discharge Date"],
    errors="coerce"
)

data["Length_of_Stay"] = (
    data["Discharge_Date"] -
    data["Admission_Date"]
).dt.days

data.loc[
    (data["Length_of_Stay"] < 0) |
    (data["Length_of_Stay"] > 365),
    "Length_of_Stay"
] = np.nan

# Gender coding
# Male = 0, Female = 1
data["Gender_Female"] = np.where(
    data["Gender"] == "Female",
    1,
    0
)

# Keep complete observations
analysis_data = data[
    [
        "High_Billing",
        "Age_Clean",
        "Gender_Female",
        "Length_of_Stay"
    ]
].dropna().copy()

# Center quantitative predictors
analysis_data["Age_Centered"] = (
    analysis_data["Age_Clean"] -
    analysis_data["Age_Clean"].mean()
)

analysis_data["LOS_Centered"] = (
    analysis_data["Length_of_Stay"] -
    analysis_data["Length_of_Stay"].mean()
)

print("\nValid observations:", len(analysis_data))

print("\nHIGH BILLING FREQUENCY TABLE")
print(
    analysis_data["High_Billing"]
    .value_counts()
    .sort_index()
)

# ============================================================
# MODEL 1: AGE ONLY
# ============================================================

X1 = sm.add_constant(
    analysis_data[
        ["Age_Centered"]
    ]
)

model1 = sm.Logit(
    analysis_data["High_Billing"],
    X1
).fit()

# ============================================================
# MODEL 2: AGE + GENDER
# ============================================================

X2 = sm.add_constant(
    analysis_data[
        [
            "Age_Centered",
            "Gender_Female"
        ]
    ]
)

model2 = sm.Logit(
    analysis_data["High_Billing"],
    X2
).fit()

# ============================================================
# MODEL 3: FINAL MODEL
# AGE + GENDER + LENGTH OF STAY
# ============================================================

X3 = sm.add_constant(
    analysis_data[
        [
            "Age_Centered",
            "Gender_Female",
            "LOS_Centered"
        ]
    ]
)

model3 = sm.Logit(
    analysis_data["High_Billing"],
    X3
).fit()

print("\n" + "=" * 75)
print("FINAL LOGISTIC REGRESSION MODEL")
print("=" * 75)

print(model3.summary())

# ============================================================
# ODDS RATIOS AND CONFIDENCE INTERVALS
# ============================================================

conf = model3.conf_int()

results = pd.DataFrame({
    "Coefficient": model3.params,
    "Odds Ratio": np.exp(model3.params),
    "P-value": model3.pvalues,
    "95% CI Lower": np.exp(conf[0]),
    "95% CI Upper": np.exp(conf[1])
})

print("\nODDS RATIOS AND 95% CONFIDENCE INTERVALS")
print(results.round(4))

# ============================================================
# CONFOUNDING CHECK
# ============================================================

age1 = model1.params["Age_Centered"]
age2 = model2.params["Age_Centered"]
age3 = model3.params["Age_Centered"]

change_gender = (
    abs(age2 - age1) /
    abs(age1)
) * 100

change_final = (
    abs(age3 - age1) /
    abs(age1)
) * 100

print("\n" + "=" * 75)
print("CONFOUNDING CHECK")
print("=" * 75)

print("Age coefficient, Model 1:", round(age1, 6))
print("Age coefficient, Model 2:", round(age2, 6))
print("Age coefficient, Model 3:", round(age3, 6))

print(
    "Percent change after adding Gender:",
    round(change_gender, 2),
    "%"
)

print(
    "Percent change after adding Gender and Length of Stay:",
    round(change_final, 2),
    "%"
)

print("\nANALYSIS COMPLETE")
#```

# Logistic Regression Output

#The analysis included **55,392 valid observations**.

#For the binary response variable:

#- **27,042 observations** had billing amounts below $25,000.
#- **28,350 observations** had billing amounts of $25,000 or more.

## Final Model Results

#| Explanatory Variable | Odds Ratio | 95% CI | p-value |
#|---|---:|---:|---:|
#| Age | 1.0002 | 0.9993–1.0010 | .700 |
#| Female Gender | 0.9811 | 0.9489–1.0143 | .262 |
#| Length of Stay | 0.9984 | 0.9965–1.0003 | .099 |

## 1. Summary of Results

#After adjusting for Gender and Length of Stay, **Age was not significantly associated with the odds of having a high billing amount**, OR = **1.0002**, 95% CI = **0.9993–1.0010**, p = **.700**.

#This means that each additional year of age was associated with almost no change in the odds of having a billing amount of $25,000 or more.

#Gender was also not significantly associated with high billing status. Female patients had slightly lower estimated odds of high billing compared with male patients, but the difference was not statistically significant, **OR = 0.9811, 95% CI = 0.9489–1.0143, p = .262**.

#Length of Stay was also not statistically significant. Each additional hospital day was associated with slightly lower estimated odds of high billing, **OR = 0.9984, 95% CI = 0.9965–1.0003, p = .099**.

#None of the explanatory variables were statistically significant at the .05 level.

## 2. Did the Results Support My Hypothesis?

#My hypothesis was that **patient Age would be associated with the likelihood of having a high Billing Amount**.

#The results did **not support this hypothesis**.

#After controlling for Gender and Length of Stay, the Age odds ratio remained extremely close to 1 and was not statistically significant:

#**OR = 1.0002, 95% CI = 0.9993–1.0010, p = .700**

#Therefore, there was no evidence that older or younger patients had meaningfully different odds of having a billing amount of $25,000 or more.

## 3. Evidence of Confounding

#I tested for confounding by adding additional explanatory variables one at a time.

#In the model containing only Age, the Age logistic regression coefficient was approximately:

#**B = 0.000160**

#After Gender was added, the Age coefficient was:

#**B = 0.000161**

#This represented a change of only approximately **0.66%**.

#After both Gender and Length of Stay were included, the Age coefficient was:

#**B = 0.000167**

#This represented a change of approximately **4.39%** from the original Age-only model.

#These changes were small and well below a commonly used practical threshold of approximately 10% for identifying meaningful confounding.

#Therefore, there was **no strong evidence that Gender or Length of Stay confounded the relationship between Age and High Billing Amount**.

# Overall Conclusion

#The logistic regression analysis found no statistically significant predictors of high billing status.

#After adjustment for Gender and Length of Stay:

#**Age:** OR = **1.0002**, 95% CI = **0.9993–1.0010**, p = **.700**

#**Female Gender:** OR = **0.9811**, 95% CI = **0.9489–1.0143**, p = **.262**

#**Length of Stay:** OR = **0.9984**, 95% CI = **0.9965–1.0003**, p = **.099**

#The findings did not support my hypothesis that Age would predict the likelihood of having a high billing amount. There was also no meaningful evidence of confounding by Gender or Length of Stay.

#These results suggest that the probability of having a billing amount of at least $25,000 is essentially unrelated to patient Age, Gender, and Length of Stay in this synthetic healthcare dataset.

## URL

#**Paste the URL of your published blog entry here.**

LOGISTIC REGRESSION: HIGH BILLING AMOUNT

Valid observations: 55392

HIGH BILLING FREQUENCY TABLE
0.0    27042
1.0    28350
Name: High_Billing, dtype: int64
Optimization terminated successfully.
         Current function value: 0.692867
         Iterations 3
Optimization terminated successfully.
         Current function value: 0.692856
         Iterations 3
Optimization terminated successfully.
         Current function value: 0.692831
         Iterations 3

FINAL LOGISTIC REGRESSION MODEL
                           Logit Regression Results                           
Dep. Variable:           High_Billing   No. Observations:                55392
Model:                          Logit   Df Residuals:                    55388
Method:                           MLE   Df Model:                            3
Date:                Tue, 06 Oct 2026   Pseudo R-squ.:               5.346e-05
Time:                        16:28:04   Log-Likelihood:                -38377.
converged:                    